# GEFCom Connecticut exploratory analysis

This notebook reads GridEval’s canonical processed CT dataset and split index files. It is read-only and does not rebuild or write processed data, split indices, or benchmark results. The weather relationship plot is descriptive exploration; the current benchmark’s default feature policy excludes weather, so this plot does not represent weather-informed model results.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "processed" / "gefcom_ct_clean.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate GridEval/data/processed/gefcom_ct_clean.parquet from the notebook's working directory.")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
ct = pd.read_parquet(PROCESSED_DIR / "gefcom_ct_clean.parquet")
if not isinstance(ct.index, pd.DatetimeIndex):
    raise ValueError("Canonical GEFCom CT data must use a DatetimeIndex.")
if not ct.index.is_monotonic_increasing or not ct.index.is_unique:
    raise ValueError("Canonical GEFCom timestamps must be unique and chronologically ordered.")
ct.head()


In [ ]:
print(f"GEFCom CT shape: {ct.shape}")
print(f"Date range: {ct.index.min()} to {ct.index.max()}")
print("Missing values:")
print(ct.isna().sum())


In [ ]:
valid = ct[["drybulb", "load"]].dropna()
poly_coefs = np.polyfit(valid["drybulb"], valid["load"], deg=2)
t_range = np.linspace(valid["drybulb"].min(), valid["drybulb"].max(), 200)
plt.figure(figsize=(9, 5))
plt.scatter(valid["drybulb"], valid["load"], alpha=0.08, color="#0284c7", s=3, label="Hourly observations")
plt.plot(t_range, poly_coefs[0] * t_range**2 + poly_coefs[1] * t_range + poly_coefs[2], color="#dc2626", linewidth=2.5, label="Descriptive quadratic fit")
plt.title("GEFCom CT: Exploratory Temperature–Load Relationship", fontsize=12)
plt.xlabel("Drybulb Temperature")
plt.ylabel("Zone Load (MW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
def load_gefcom_split(split_name):
    frame = pd.read_parquet(PROCESSED_DIR / f"gefcom_{split_name}_idx.parquet")
    return pd.DatetimeIndex(frame["timestamp"], name="timestamp")

train_idx = load_gefcom_split("train")
val_idx = load_gefcom_split("val")
test_idx = load_gefcom_split("test")
train_gefcom = ct.loc[train_idx]
val_gefcom = ct.loc[val_idx]
test_gefcom = ct.loc[test_idx]

assert train_gefcom.index.equals(train_idx)
assert val_gefcom.index.equals(val_idx)
assert test_gefcom.index.equals(test_idx)
assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx))
assert [len(train_idx), len(val_idx), len(test_idx)] == [26298, 8758, 8782]
assert train_idx.max() < val_idx.min() < val_idx.max() < test_idx.min()
for label, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    print(f"GEFCom {label}: {len(idx):,} observations ({idx.min()} to {idx.max()})")
print("Split identity, ordering, non-overlap, and expected counts verified against canonical split files.")


## Safe rerun note

All cells above read the canonical artifacts and perform exploratory analysis. They intentionally contain no `to_parquet`, `to_csv`, or other writes to GridEval data or result files. The weather plot is descriptive only and should not be cited as evidence that weather features were used in the current benchmark.
